# HLS-CMDS - Nhan dang am tim/phoi BAT THUONG (noi tiep phan tach am)

Pipeline: **hon hop -> SPMamba (tach) -> 2 bo phan loai (nhan dang)**

- ch0 = tim  -> 10 lop benh ly + nhi phan Normal/Abnormal
- ch1 = phoi -> 6 lop benh ly + nhi phan Normal/Abnormal

### Luu y quan trong ve du lieu
Chi co **50 ban ghi doc lap moi nhanh** (lop nho nhat: S4 co 2 ban ghi).
- Moi phep chia deu theo **ban ghi**, khong theo cua so audio, neu khong se ro ri du lieu.
- **File .wav bi .gitignore** nen clone repo KHONG co audio. Bat buoc phai upload zip
  dataset HLS-CMDS len Drive (xem `DRIVE_DATASET_ZIP` o Cell 1).
- So lieu de BAO CAO lay tu **cross-validation long** (Cell 9), khong phai tu mot lan chia.
- Ket qua end-to-end CHINH la che do `cascade-synth`. Che do `cascade-real` co ro ri
  du lieu, chi dung tham khao.

### Thu tu chay
Cell 1 -> 2 -> 3 -> 4 -> 5 (kiem tra nhanh) -> 6 (train tim) -> 7 (train phoi)
-> 8 (danh gia). **Cell 9** la cross-validation, chay them de lay so lieu bao cao.

In [ ]:
# === Cell 1: Cau hinh duong dan + kiem tra GPU ===
from google.colab import drive
from pathlib import Path
import os, subprocess

drive.mount('/content/drive')

REPO_URL = "https://github.com/hoanganh0106/spmamba.git"
GIT_ROOT = "/content/spmamba"
REPO_DIR = "/content/spmamba/hls"
ENV_DIR  = "/content/spmamba-env"     # dung chung env voi train_hls_colab.ipynb
MM       = "/content/bin/micromamba"

# --- 1) H5 tach am: de KE THUA dung phan hoach nguon -> chong ro ri du lieu ---
DRIVE_H5_ZIP   = "/content/drive/MyDrive/data_hls.zip"
DRIVE_H5_PATH  = ""                    # hoac tro thang toi file .h5 tren Drive
LOCAL_H5       = "/content/data_hls.h5"

# --- 2) Dataset goc HLS-CMDS (chua file .wav) ---
# BAT BUOC: file *.wav bi .gitignore nen clone repo chi co 3 file CSV, khong co audio.
# Repo VAN chua thu muc HLS-CMDS/HS/HS (chi co README) nen Cell 4 phai loai tru no.
DRIVE_DATASET_ZIP = "/content/drive/MyDrive/HLS-CMDS.zip"
DATASET_ROOT      = ""                 # de trong = Cell 4 tu tim sau khi giai nen
RAW_EXTRACT_DIR   = "/content/hls_raw"

# --- 3) H5 phan loai (Cell 4 se dung neu chua co) ---
LOCAL_CLS_H5   = "/content/data_hls_cls.h5"
DRIVE_CLS_H5   = ""                    # neu da build san tu truoc thi tro vao day

CONFIG_PATH    = f"{REPO_DIR}/configs/hls-cls-t4.yml"
CKPT_DIR       = "/content/drive/MyDrive/SPMamba_checkpoints/HLS-cls"
# Checkpoint SPMamba da train o buoc tach am (dung cho danh gia cascade)
SEP_CKPT_DIR   = "/content/drive/MyDrive/SPMamba_checkpoints/SPMamba-HLS-HeartLung-T4"

WANDB_PROJECT  = "spmamba"

WANDB_API_KEY = ""
try:
    from google.colab import userdata
    WANDB_API_KEY = userdata.get("WANDB_API_KEY") or ""
except Exception as e:
    print("[wandb] Khong doc duoc Colab Secret:", e)
WANDB_API_KEY = WANDB_API_KEY or os.environ.get("WANDB_API_KEY", "")
if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    print("[wandb] Da co WANDB_API_KEY")
else:
    print("[wandb] Chua co WANDB_API_KEY -> van train binh thuong, chi khong log W&B")

Path(CKPT_DIR).mkdir(parents=True, exist_ok=True)
for k in ["REPO_DIR", "ENV_DIR", "LOCAL_H5", "DRIVE_DATASET_ZIP", "LOCAL_CLS_H5",
          "CONFIG_PATH", "CKPT_DIR", "SEP_CKPT_DIR"]:
    print(f"{k} = {globals()[k]}")
subprocess.run("nvidia-smi", shell=True, check=False)

In [ ]:
# === Cell 2: Moi truong (dung lai env cua phan tach am) ===
# Neu da chay train_hls_colab.ipynb trong cung phien Colab thi env da san sang,
# de INSTALL_DEPS = False cho nhanh.
import os, subprocess, shutil
from pathlib import Path

os.environ["MAMBA_ROOT_PREFIX"] = "/content/micromamba-root"
os.environ["PATH"] = "/content/bin:" + os.environ.get("PATH", "")

INSTALL_DEPS = True
FORCE_RECREATE_ENV = False

def run(cmd, cwd=None, check=True):
    print(f"\n[cmd] {cmd}", flush=True)
    return subprocess.run(cmd, shell=True, cwd=cwd, env=os.environ.copy(), text=True, check=check)

Path('/content/bin').mkdir(parents=True, exist_ok=True)
if not Path(MM).exists():
    run("curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /content bin/micromamba")
run(f"{MM} --version")

if FORCE_RECREATE_ENV and Path(ENV_DIR).exists():
    shutil.rmtree(ENV_DIR)
if not Path(ENV_DIR).exists():
    run(f"{MM} create -p {ENV_DIR} python=3.10 --yes -q")

if INSTALL_DEPS:
    run(f"{MM} run -p {ENV_DIR} python -m pip install -U pip -q")
    run(f"{MM} run -p {ENV_DIR} pip install 'setuptools<70' typeguard==2.13.3 h5py pytorch-lightning==2.0.2 einops soundfile librosa fast-bss-eval speechbrain==0.5.14 hydra-core rich torch-complex packaging wandb pyyaml -q")
    run(f"{MM} run -p {ENV_DIR} pip install --force-reinstall scipy -q")
    run(f"{MM} run -p {ENV_DIR} pip install torch-mir-eval torch-optimizer transformers -q")
    run(f"{MM} run -p {ENV_DIR} pip install torch==2.5.0 torchaudio==2.5.0 torchvision==0.20.0 --index-url https://download.pytorch.org/whl/cu124 --force-reinstall -q")
    # mamba_ssm chi can cho SPMamba (danh gia cascade o Cell 8). Bo phan loai KHONG
    # dung toi - look2hear.models nap SPMamba kieu luoi nen Cell 5/6/7 chay duoc
    # ngay ca khi dong duoi day that bai.
    run(f"{MM} run -p {ENV_DIR} pip install 'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.0/causal_conv1d-1.6.0+cu12torch2.5cxx11abiFALSE-cp310-cp310-linux_x86_64.whl' 'https://github.com/state-spaces/mamba/releases/download/v2.3.0/mamba_ssm-2.3.0+cu12torch2.5cxx11abiFALSE-cp310-cp310-linux_x86_64.whl' --no-deps --force-reinstall -q", check=False)

run(f"{MM} run -p {ENV_DIR} python -c \"import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())\"")
run(f"{MM} run -p {ENV_DIR} python -c \"import h5py, yaml, soundfile; print('h5py/yaml/soundfile OK')\"")
run("nvidia-smi", check=False)

In [ ]:
# === Cell 3: Clone/cap nhat repo ===
from pathlib import Path
import subprocess, os

def run_list(cmd, cwd=None, check=True):
    print("\n[cmd]", " ".join(cmd), flush=True)
    return subprocess.run(cmd, cwd=cwd, env=os.environ.copy(), text=True, check=check)

repo = Path(REPO_DIR)
if repo.exists():
    run_list(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
else:
    run_list(["git", "clone", REPO_URL, REPO_DIR])
run_list(["git", "-C", REPO_DIR, "log", "--oneline", "-3"], check=False)

In [ ]:
# === Cell 4: Chuan bi du lieu + dung H5 phan loai ===
# Ba viec:
#   1. Dua data_hls.h5 (tach am) ve /content  -> de KE THUA phan hoach nguon
#   2. Giai nen dataset goc HLS-CMDS          -> lay file .wav (repo khong co)
#   3. Chay build_hls_cls_h5.py               -> /content/data_hls_cls.h5
from pathlib import Path
import shutil, zipfile, subprocess, os, h5py

REBUILD_CLS_H5 = False   # dat True neu muon dung lai tu dau

# --- 1) data_hls.h5 ---
if not Path(LOCAL_H5).exists():
    if DRIVE_H5_PATH and Path(DRIVE_H5_PATH).exists():
        shutil.copy2(DRIVE_H5_PATH, LOCAL_H5); print("[data] copy H5 tach am tu Drive")
    elif DRIVE_H5_ZIP and Path(DRIVE_H5_ZIP).exists():
        with zipfile.ZipFile(DRIVE_H5_ZIP) as zf:
            members = [n for n in zf.namelist()
                       if n.lower().endswith((".h5", ".hdf5")) and not n.endswith("/")]
            if not members:
                raise RuntimeError(f"Zip khong co file .h5: {DRIVE_H5_ZIP}")
            with zf.open(members[0]) as src, open(LOCAL_H5, "wb") as dst:
                shutil.copyfileobj(src, dst)
        print("[data] da giai nen H5 tach am")
    else:
        raise FileNotFoundError(
            "Khong thay data_hls.h5. Can no de ke thua phan hoach nguon "
            "(chong ro ri du lieu). Sua DRIVE_H5_ZIP / DRIVE_H5_PATH o Cell 1."
        )
print("[data] H5 tach am:", LOCAL_H5)

# --- 2) Dataset goc HLS-CMDS ---
# QUAN TRONG: chi tim trong thu muc giai nen, KHONG quet ca /content.
#   - repo clone o /content/spmamba CO thu muc HLS-CMDS/HS/HS nhung 0 file .wav
#     (vi *.wav nam trong .gitignore) -> neu nhan nham se chet o assert ben duoi
#   - /content/drive la Drive gan qua FUSE, quet de quy se rat cham
#   - /content/spmamba-env co hang tram nghin file
def find_dataset_root(bases=(RAW_EXTRACT_DIR, "/content/HLS-CMDS")):
    for base in bases:
        if not Path(base).exists():
            continue
        for hs in Path(base).rglob("HLS-CMDS/HS/HS"):
            root = hs.parent.parent
            n_h = len(list(hs.glob("*.wav")))
            n_l = len(list((root / "LS/LS").glob("*.wav")))
            print(f"[data] ung vien: {root}  (HS={n_h} wav, LS={n_l} wav)")
            if n_h == 50 and n_l == 50:      # chi nhan khi CO du audio
                return str(root)
    return ""

if not DATASET_ROOT:
    DATASET_ROOT = find_dataset_root()
if not DATASET_ROOT:
    if not (DRIVE_DATASET_ZIP and Path(DRIVE_DATASET_ZIP).exists()):
        raise FileNotFoundError(
            f"Khong thay dataset goc HLS-CMDS co du 50+50 file .wav.\n"
            f"  File *.wav bi .gitignore nen clone repo KHONG co audio.\n"
            f"  Hay upload zip dataset len Drive va sua DRIVE_DATASET_ZIP (Cell 1).\n"
            f"  Dang tim zip tai: {DRIVE_DATASET_ZIP}"
        )
    print("[data] giai nen dataset goc:", DRIVE_DATASET_ZIP)
    with zipfile.ZipFile(DRIVE_DATASET_ZIP) as zf:
        zf.extractall(RAW_EXTRACT_DIR)
    DATASET_ROOT = find_dataset_root()
    if not DATASET_ROOT:
        raise RuntimeError(
            f"Giai nen xong nhung khong tim thay HLS-CMDS co du 50+50 wav trong "
            f"{RAW_EXTRACT_DIR}. Kiem tra lai cau truc zip."
        )
print("[data] DATASET_ROOT =", DATASET_ROOT)

# --- 3) Dung H5 phan loai ---
if DRIVE_CLS_H5 and Path(DRIVE_CLS_H5).exists() and not REBUILD_CLS_H5:
    shutil.copy2(DRIVE_CLS_H5, LOCAL_CLS_H5); print("[data] copy H5 phan loai tu Drive")
elif REBUILD_CLS_H5 or not Path(LOCAL_CLS_H5).exists():
    cmd = [MM, "run", "-p", ENV_DIR, "python", "-u", "build_hls_cls_h5.py",
           "--dataset-root", DATASET_ROOT,
           "--align-split-with", LOCAL_H5,
           "--out", LOCAL_CLS_H5]
    env = os.environ.copy(); env["PYTHONPATH"] = REPO_DIR
    print("[build]", " ".join(cmd))
    subprocess.run(cmd, cwd=REPO_DIR, env=env, check=True)
else:
    print("[data] da co", LOCAL_CLS_H5)

with h5py.File(LOCAL_CLS_H5, "r") as f:
    print("\n[h5 phan loai] attrs:", dict(f.attrs))
    for b in ("heart", "lung"):
        print(f"  /{b}: audio={f[b]['audio'].shape}, {f[b].attrs['n_classes']} lop")

In [ ]:
# === Cell 5: Kiem tra nhanh truoc khi train (nen chay) ===
# Kiem tra chi so, bang nhan, shape mo hinh, tinh roi nhau cua split va mot vong
# train ngan. Vai chuc giay - re hon nhieu so voi phat hien loi sau mot gio train.
import os, subprocess

cmd = [MM, "run", "-p", ENV_DIR, "python", "-u", "smoke_hls_cls.py",
       "--h5", LOCAL_CLS_H5, "--dataset_root", DATASET_ROOT]

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"; env["PYTHONPATH"] = REPO_DIR

proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=env, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError("Smoke test KHONG dat - sua loi truoc khi train (xem cac dong [FAIL])")
print("\n[smoke] tat ca dat - co the chay Cell 6/7 de train")

In [ ]:
# === Cell 6: Train bo phan loai TIM ===
import os, subprocess
from pathlib import Path

BRANCH = "heart"
out_dir = f"{CKPT_DIR}/{BRANCH}"
Path(out_dir).mkdir(parents=True, exist_ok=True)

cmd = [MM, "run", "-p", ENV_DIR, "python", "-u", "train_hls_cls.py",
       "--branch", BRANCH, "--h5", LOCAL_CLS_H5,
       "--conf_dir", CONFIG_PATH, "--out_dir", out_dir]
if WANDB_API_KEY:
    cmd += ["--wandb", "--wandb_project", WANDB_PROJECT,
            "--wandb_run_name", f"HLS-cls-{BRANCH}"]

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"; env["PYTHONPATH"] = REPO_DIR
if WANDB_API_KEY:
    env["WANDB_API_KEY"] = WANDB_API_KEY

print("[train]", " ".join(cmd))
proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=env, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
except KeyboardInterrupt:
    proc.terminate(); raise
# Bao loi ro rang thay vi de cell "xanh" roi Cell 8 chet vi thieu best.pt
if proc.returncode != 0:
    raise RuntimeError(f"Train {BRANCH} THAT BAI - exit code {proc.returncode}")
print(f"\n[train] {BRANCH} xong, checkpoint o {out_dir}")

In [ ]:
# === Cell 7: Train bo phan loai PHOI ===
import os, subprocess
from pathlib import Path

BRANCH = "lung"
out_dir = f"{CKPT_DIR}/{BRANCH}"
Path(out_dir).mkdir(parents=True, exist_ok=True)

cmd = [MM, "run", "-p", ENV_DIR, "python", "-u", "train_hls_cls.py",
       "--branch", BRANCH, "--h5", LOCAL_CLS_H5,
       "--conf_dir", CONFIG_PATH, "--out_dir", out_dir]
if WANDB_API_KEY:
    cmd += ["--wandb", "--wandb_project", WANDB_PROJECT,
            "--wandb_run_name", f"HLS-cls-{BRANCH}"]

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"; env["PYTHONPATH"] = REPO_DIR
if WANDB_API_KEY:
    env["WANDB_API_KEY"] = WANDB_API_KEY

print("[train]", " ".join(cmd))
proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=env, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
except KeyboardInterrupt:
    proc.terminate(); raise
# Bao loi ro rang thay vi de cell "xanh" roi Cell 8 chet vi thieu best.pt
if proc.returncode != 0:
    raise RuntimeError(f"Train {BRANCH} THAT BAI - exit code {proc.returncode}")
print(f"\n[train] {BRANCH} xong, checkpoint o {out_dir}")

In [ ]:
# === Cell 8: Danh gia - clean / cascade-synth / cascade-real ===
# cascade-synth la KET QUA CHINH (khong ro ri).
# cascade-real chi tham khao: cac take H####/L#### trung lap voi du lieu train.
import os, subprocess

HEART_CKPT = f"{CKPT_DIR}/heart/best.pt"
LUNG_CKPT  = f"{CKPT_DIR}/lung/best.pt"
OUT_JSON   = f"{CKPT_DIR}/ket_qua_danh_gia.json"

for p in (HEART_CKPT, LUNG_CKPT):
    assert os.path.exists(p), f"Chua co {p} - chay Cell 6/7 truoc"

cmd = [MM, "run", "-p", ENV_DIR, "python", "-u", "test_hls_cls.py",
       "--mode", "all",
       "--heart_ckpt", HEART_CKPT, "--lung_ckpt", LUNG_CKPT,
       "--cls_h5", LOCAL_CLS_H5,
       "--sep_ckpt", SEP_CKPT_DIR, "--h5", LOCAL_H5,
       "--dataset_root", DATASET_ROOT,
       "--out_json", OUT_JSON]

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"; env["PYTHONPATH"] = REPO_DIR

print("[eval]", " ".join(cmd))
proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=env, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
print(f"\n[eval] exit code {proc.returncode}  (so lieu day du o {OUT_JSON})")
if proc.returncode != 0:
    print("[eval] CANH BAO: co che do danh gia that bai - xem log o tren")

In [ ]:
# === Cell 9: Cross-validation long 5 fold - SO LIEU DE BAO CAO ===
# Mot lan chia chi cho ~5 ban ghi test nen rat nhieu. Cross-validation LONG:
#   fold k      -> tap BAO CAO (khong bao gio dung de chon epoch)
#   fold (k+1)  -> chon epoch tot nhat + dung som
#   con lai     -> train
# Ban ghi thuoc pool test cua bo tach am co fold = -1 va bi loai khoi CV, de
# khong lam hong thi nghiem cascade.
import os, subprocess, json, statistics
from pathlib import Path

BRANCHES = ["heart", "lung"]
N_FOLDS = 5
cv_root = f"{CKPT_DIR}/cv"
failed = []

for branch in BRANCHES:
    for fold in range(N_FOLDS):
        out_dir = f"{cv_root}/{branch}"
        cmd = [MM, "run", "-p", ENV_DIR, "python", "-u", "train_hls_cls.py",
               "--branch", branch, "--h5", LOCAL_CLS_H5,
               "--conf_dir", CONFIG_PATH, "--out_dir", out_dir, "--fold", str(fold)]
        env = os.environ.copy()
        env["PYTHONUNBUFFERED"] = "1"; env["PYTHONPATH"] = REPO_DIR
        print(f"\n{'='*60}\n  {branch} - fold {fold}\n{'='*60}", flush=True)
        proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=env, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end="", flush=True)
        proc.wait()
        if proc.returncode != 0:
            failed.append((branch, fold, proc.returncode))
            print(f"[cv] THAT BAI: {branch} fold {fold} (exit {proc.returncode})")

if failed:
    print("\n[cv] CANH BAO - cac lan chay that bai:", failed)

print(f"\n\n{'='*66}\n  TONG HOP CROSS-VALIDATION ({N_FOLDS} fold)\n{'='*66}")
for branch in BRANCHES:
    rows = []
    for fold in range(N_FOLDS):
        p = Path(f"{cv_root}/{branch}/fold{fold}/summary.json")
        if p.exists():
            rows.append(json.loads(p.read_text(encoding="utf-8")))
    if not rows:
        print(f"  {branch}: khong co summary.json"); continue

    biased = [r for r in rows if r.get("selection_set_equals_eval_set")]
    if biased:
        print(f"  [{branch}] CANH BAO: {len(biased)} fold co tap chon = tap bao cao "
              "-> so lieu thien lech lac quan, KHONG dung de bao cao")

    def agg(getter, label):
        vals = [v for v in (getter(r) for r in rows) if v is not None]
        if not vals:
            print(f"    {label:<26s} n/a"); return
        sd = statistics.stdev(vals) if len(vals) > 1 else 0.0
        print(f"    {label:<26s} {statistics.mean(vals):.4f} +/- {sd:.4f}   {[round(v,3) for v in vals]}")

    n_rec = sum(r.get("n_eval_records", 0) for r in rows)
    print(f"\n  [{branch}]  {len(rows)}/{N_FOLDS} fold, tong {n_rec} ban ghi duoc cham")
    agg(lambda r: r["binary"]["macro_f1"],    "macro-F1 nhi phan")
    agg(lambda r: r["binary"]["sensitivity"], "do nhay (sensitivity)")
    agg(lambda r: r["binary"]["specificity"], "do dac hieu (specificity)")
    agg(lambda r: r["binary"]["auroc"],       "AUROC nhi phan")
    agg(lambda r: r["multi_f1"],              "macro-F1 da lop")
    agg(lambda r: r["multi_balanced_acc"],    "balanced-acc da lop")